# ⚠️ Notebook 4 – Churn Prediction
**RetailPulse | XGBoost + SHAP | Target AUC-ROC ≥ 0.88**

Steps:
1. Build churn labels (90-day inactivity)
2. Feature engineering (RFM + behavioural)
3. XGBoost with Optuna tuning
4. SHAP explainability
5. MLflow tracking


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import mlflow
import os
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '../..')
plt.style.use('dark_background')
os.environ['MLFLOW_TRACKING_URI'] = '../../mlruns'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
print('Libraries loaded ✅')

## 1. Load Features

In [ ]:
churn_df = pd.read_parquet('../../data/processed/churn_features.parquet')
print(f'Shape: {churn_df.shape}')
print(f'Churn rate: {churn_df["churned"].mean():.2%}')
churn_df.head()

## 2. Exploratory Analysis – Churn vs Non-Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['Recency','Frequency','Monetary']):
    churn_df[churn_df['churned']==0][col].hist(ax=ax, alpha=0.6, color='#10b981', label='Active', bins=30)
    churn_df[churn_df['churned']==1][col].hist(ax=ax, alpha=0.6, color='#ef4444', label='Churned', bins=30)
    ax.set_title(f'{col} Distribution')
    ax.legend()
plt.tight_layout()
plt.savefig('../../reports/churn_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Train XGBoost Model

In [ ]:
from src.models.churn import train_churn_model, predict_churn_risk

results = train_churn_model(churn_df, model_dir='../../models/', n_trials=15)
print(f'\n=== Results ===')
for k,v in results['metrics'].items():
    print(f'  {k}: {v}')

## 4. SHAP Feature Importance

In [ ]:
from src.models.churn import get_shap_summary, FEATURE_COLS
import pandas as pd

shap_summary = get_shap_summary(results['shap_values'], FEATURE_COLS)
print(shap_summary)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(shap_summary['feature'], shap_summary['mean_abs_shap'], color='#ef4444', alpha=0.8)
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('SHAP Feature Importance – Churn Prediction', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../../reports/churn_shap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Score All Customers & Risk Tiers

In [ ]:
scored = predict_churn_risk(results['model'], churn_df)
scored.to_parquet('../../data/processed/churn_scored.parquet', index=False)

print(scored['churn_risk'].value_counts())
print(f'\nHigh risk customers: {(scored["churn_risk"]=="High").sum():,}')
print(f'Saved to data/processed/churn_scored.parquet ✅')